In [ ]:
'''
A multi-objective optimization problem for the conceptual design of an aircraft using evolutionary algorithms. The aim is to generate aircraft configurations that reduce aerodynamic drag, provide sufficient fuel volume inside the wings, and maximize cargo capacity.
'''

In [ ]:
# Minimize aerodynamic drag.
# Maximize the available fuel volume inside the wings.
# Maximize the available cargo space.

In [1]:
#

In [1]:
#b -> wing span
#Sw-> wing area
#AR wing aspect ratio
#^ -> wing sweep angle
#tc -> wing tickness - to - chord ratio
#Lf fuselage length
#Df fuselage diametre
# Vc available cargo volume
# Vf fuel volume stored inside wings
#St tail surface area 
# xe engine placement

In [ ]:
#AR=b^2/Sw
#

In [2]:
'''
OBJECTIVE FCT 
The first objective is to minimize the drag coefficient:

min f1(x)=CD(x) 

The second objective is to maximize the fuel volume inside the wings:

max f2(x) = Vfuel(x)

The third objective is to maximize the available cargo volume:

max f3(x)=Vcargo(x)

Since many evolutionary algorithms are formulated as minimization algorithms, the problem can be written as:

min F(x)=[CD(x),-Vfuel(x),-Vcargo(x)]

where:
CD(x) is the aerodynamic drag coefficient;
Vfuel(x) is the fuel volume stored inside the wings;
Vcargo(x) is the available cargo space.
'''

'\nOBJECTIVE FCT \n'

In [5]:
'''
Simplified aerodynamic model

CD(x)=CD0(x)+CDi(x)
zero-lift drag coef + induced drag coef

CDi(x) = CL^2/pi*e*AR 
CL - lift coefficient
e - Oswald efficiency factor
AR - wing aspect ratio

CL = 2W/ro*V^2*Sw
W - aircraft weight
ro -air density
V - flight velocity
Sw - is the wing area.

D(x) drag force = 1/2 * ro*V^2 * Sw * CD(x)
'''

'\nSimplified aerodynamic model\n\nCD(x)=CD0(x)+CDi(x)\nzero-lift drag coef + induced drag coef\n\nCDi(x) = CL^2/pi*e*AR \nCL - lift coefficient\ne - Oswald efficiency factor\nAR - wing aspect ratio\n\nCL = 2W/ro*V^2*Sw\nW - aircraft weight\nro -air density\nV - flight velocity\nSw - is the wing area.\n'

In [ ]:
'''
Fuel volume model

Vfuel (x)= kj * Sw * c(with a dash on top) * tc
kf - fuel volume efficiency coef
Sw - wing area
c(with a dash on top) mean aerodynamic chord 
tx - wing thickness - to - chord ratio

 c(with a dash on top)=Sw/b
 =>
 Vfuel(x)=kf*Sw*(Sw/b)tc
 
'''

In [6]:
'''
Cargo volume model

Vcargo(x)=kc * Lf * Df^2
kc - cargo-space efficiency coef 
Lf - fuselage length
Df - fuselage diameter 
'''

'\nCargo volume model\n\nVcargo(x)=kc * Lf * Df^2\n'

In [8]:
'''
Lift constraint :

L(x)>=W(x)

L(x)=1/2 * ro* V^2 * Sw * CL

1. min fuel cap -> Vfuel(x)>= Vfuel min
2. min cargo cap -> Vcargo(x)>= Vcargo min
3. structural constraint -> sigma(x)<=sigma max
4. wing area bounds -> Sw min <= Sw(x) <= Sw max
5. aspect ratio bounds -> AR min<=AR(x) <= AR max
6. stability constraint SM(x)>=SM min
'''

'\nLift constraint :\n\nL(x)>=W(x)\n\nL(x)=1/2 * ro* V^2 * Sw * CL\n\n1. min fuel cap -> Vfuel(x)>= Vfuel min\n'

In [ ]:
'''
evolutionary algorithm formulation 


can use:
Genetic Algorithm
NSGA-II
Differential Evolution
Multi-Objective Particle Swarm Optimization
Evolution Strategies

- individual = aircraft design 
- N population size
- each individual is evaluated using F(xi)=[CD(xi)-Vfuel(xi)-Vcargo(xi)]
- apply selection, crossover, mutation, and replacement op in order to evolve 

'''

In [9]:
'''
Constraint handling 

penalized obj -> Fp(x)=F(x)+ lambda*P(x)
Fp(x)= penalized obj vector
F(x) = original obj
lambda= penalty coeff
P(x) = total constraint violation

total penalty fct = P(x) = sum for j=1->m of (max(0,gj(x))^2 ) where gj(x)<=0 represents j th inequality constraint 
'''

'\nConstraint handling \n\npenalized obj -> Fp(x)=F(x)+ lambda*P(x)\nFp(x)= penalized obj vector\nF(x) = original obj\nlambda= penalty coeff\nP(x) = total constraint violation\n\ntotal penalty fct = P(x) = sum for j=1->m of (max(0,gj(x))^2 )\n'

In [10]:
'''
Pareto Optimality

xa -> dominates xb if 
fi(xa)<=fi(xb) for all i 
and exists i such that fi(xa)<fi(xb)

PF={x from X with the property that doesn't exist any y from X such that y dominates x}
'''

'\nPareto Optimality\n\nxa -> dominates xb if \n'

In [11]:
'''
generic evolutionary algorithm procedure

 Initialize a population of aircraft designs.
Evaluate aerodynamic drag, fuel volume, and cargo volume for each design.
Check constraints and apply penalties if necessary.
Select the best individuals based on Pareto dominance and diversity.
Apply crossover to generate new aircraft designs.
Apply mutation to introduce geometric variation.
Replace part or all of the population with the new individuals.
Repeat the process until a stopping criterion is met.
Return the Pareto-optimal set of aircraft designs.


stopping criteria :

t>=Tmax OR deltaPF<epsilon
t- current generation
Tmax - maximum nb of generations
delta PF - change in Pareto front 
epsilon - convergence threshold
'''

'\ngeneric evolutionary algorithm procedure\n\n Initialize a population of aircraft designs.\nEvaluate aerodynamic drag, fuel volume, and cargo volume for each design.\nCheck constraints and apply penalties if necessary.\nSelect the best individuals based on Pareto dominance and diversity.\nApply crossover to generate new aircraft designs.\nApply mutation to introduce geometric variation.\nReplace part or all of the population with the new individuals.\nRepeat the process until a stopping criterion is met.\nReturn the Pareto-optimal set of aircraft designs.\n\n\nstopping criteria \n'

In [ ]:
"""EXAMPLE"""

In [12]:
import numpy as np

# Constants used in the simplified model
rho = 1.225          # air density at sea level [kg/m^3]
V = 230.0            # cruise speed [m/s]
e = 0.8              # Oswald efficiency factor
W = 7.0e5            # aircraft weight [N]
k_f = 0.45           # fuel volume efficiency coefficient
k_c = 0.60           # cargo volume efficiency coefficient

# Minimum requirements
V_fuel_min = 80.0    # minimum fuel volume [m^3]
V_cargo_min = 120.0  # minimum cargo volume [m^3]
AR_min = 6.0
AR_max = 14.0
S_w_min = 80.0
S_w_max = 300.0

In [13]:
def evaluate_aircraft(x):
    """
    Evaluate a simplified aircraft design.
    Design vector: x = [b, S_w, sweep, t_c, L_f, D_f]
    """
    b, S_w, sweep, t_c, L_f, D_f = x
    AR = b**2 / S_w
    C_L = 2 * W / (rho * V**2 * S_w)
    C_D0 = 0.018 + 0.002 * (D_f / L_f) + 0.0001 * abs(sweep)
    C_Di = C_L**2 / (np.pi * e * AR)
    C_D = C_D0 + C_Di
    c_bar = S_w / b
    V_fuel = k_f * S_w * c_bar * t_c
    V_cargo = k_c * L_f * D_f**2
    return C_D, V_fuel, V_cargo, AR

In [14]:
def constraint_penalty(x):
    """Compute a simple penalty for constraint violations."""
    C_D, V_fuel, V_cargo, AR = evaluate_aircraft(x)
    b, S_w, sweep, t_c, L_f, D_f = x
    violations = [
        max(0, V_fuel_min - V_fuel),
        max(0, V_cargo_min - V_cargo),
        max(0, AR_min - AR),
        max(0, AR - AR_max),
        max(0, S_w_min - S_w),
        max(0, S_w - S_w_max),
    ]
    return sum(v**2 for v in violations)

In [15]:
def objective_function(x, penalty_weight=1e3):
    """
    Multi-objective function converted into a penalized vector.
    Objectives: minimize drag, maximize fuel volume, maximize cargo volume.
    """
    C_D, V_fuel, V_cargo, AR = evaluate_aircraft(x)
    penalty = constraint_penalty(x)
    return np.array([
        C_D + penalty_weight * penalty,
        -V_fuel + penalty_weight * penalty,
        -V_cargo + penalty_weight * penalty,
    ])

In [16]:
# Example candidate aircraft design
# x = [wing span, wing area, sweep angle, thickness-to-chord ratio, fuselage length, fuselage diameter]
x_example = np.array([42.0, 140.0, 25.0, 0.12, 38.0, 4.5])

C_D, V_fuel, V_cargo, AR = evaluate_aircraft(x_example)
F = objective_function(x_example)

print("Example aircraft design:")
print(f"Drag coefficient C_D: {C_D:.5f}")
print(f"Fuel volume: {V_fuel:.2f} m^3")
print(f"Cargo volume: {V_cargo:.2f} m^3")
print(f"Aspect ratio: {AR:.2f}")
print(f"Objective vector: {F}")

Example aircraft design:
Drag coefficient C_D: 0.02149
Fuel volume: 25.20 m^3
Cargo volume: 461.70 m^3
Aspect ratio: 12.60
Objective vector: [3003040.02148882 3003014.8        3002578.3       ]


In [ ]:
#NSGA - II
# well documented , easy to implement 
#provides non - dominated sorting
# crowding distance
# simulated binary crossover 